In [20]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [21]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/isic-flagship-project")
os.chdir(PROJECT_ROOT)

print("Working in:", PROJECT_ROOT)

Working in: /content/drive/MyDrive/isic-flagship-project


In [ ]:
from pathlib import Path

folders = [
    ".github/workflows",
    "tests",
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

In [12]:
%%writefile pyproject.toml
[tool.black]
line-length = 100
target-version = ["py311"]
exclude = '''
/(
    \.git
  | \.venv
  | venv
  | env
  | notebooks
  | data
  | logs
)/
'''

[tool.ruff]
line-length = 100
target-version = "py311"
exclude = [
  ".git",
  ".venv",
  "venv",
  "env",
  "notebooks",
  "data",
  "logs",
  "*.ipynb",
  "Rag_assistant.py",
  "Setup.py",
  "agent_evaluation_suite.py",
  "copilot_studio_agent.py",
  "docker_api_deployment.py",
  "fastapi_inference_backend.py",
  "inference_engine_refactor.py",
  "production_backend_features.py",
  "prompt_versioning.py",
  "real_ensemble_integration.py",
]

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B"]
ignore = [
  "E501",
  "B008"
]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["."]
addopts = "-ra"

Overwriting pyproject.toml


In [ ]:
%%writefile tests/conftest.py
"""Test configuration that prevents real external calls in CI."""

import os

os.environ.setdefault("APP_NAME", "ISIC2024-Flagship")
os.environ.setdefault("API_VERSION", "v1")
os.environ.setdefault("MODEL_VERSION", "2024-ensemble-2models")
os.environ.setdefault("DEBUG", "False")
os.environ.setdefault("DATABASE_URL", "sqlite:///./test_isic.db")
os.environ.setdefault("SECRET_KEY", "ci-test-secret-key")
os.environ.setdefault("POWER_AUTOMATE_URL", "")
os.environ.setdefault("OPENAI_API_KEY", "sk-ci-dummy-not-used")
os.environ.setdefault("CI", "true")

Writing tests/conftest.py


In [ ]:
%%writefile tests/test_app_import.py
def test_app_imports():
    from src.api.main import app

    routes = {route.path for route in app.routes}

    assert "/" in routes
    assert "/api/v1/health" in routes
    assert "/api/v1/predict" in routes

Writing tests/test_app_import.py


In [ ]:
%%writefile tests/test_health.py
from fastapi.testclient import TestClient

from src.api.main import app


def test_health_endpoint():
    client = TestClient(app)

    response = client.get("/api/v1/health")

    assert response.status_code == 200
    body = response.json()

    assert body.get("status") in {"healthy", "ok"}
    assert "model_version" in body

Writing tests/test_health.py


In [ ]:
%%writefile tests/test_predict_invalid_upload.py
from fastapi.testclient import TestClient

from src.api.main import app


def test_predict_rejects_non_image_upload():
    client = TestClient(app)

    response = client.post(
        "/api/v1/predict",
        files={"file": ("not-an-image.txt", b"not image bytes", "text/plain")},
    )

    assert response.status_code == 400
    assert "image" in response.json()["detail"].lower()

Writing tests/test_predict_invalid_upload.py


In [ ]:
%%writefile tests/test_rag_and_agent.py
from fastapi.testclient import TestClient

from src.api.main import app


def _paths():
    return {route.path for route in app.routes}


def test_rag_chat_endpoint_if_registered(monkeypatch):
    if "/api/v1/chat" not in _paths():
        return

    try:
        import src.api.rag_routes as rag_routes
    except ModuleNotFoundError:
        return

    if hasattr(rag_routes, "rag_engine"):
        monkeypatch.setattr(
            rag_routes.rag_engine,
            "ask",
            lambda question: f"Mocked project context answer for: {question}",
            raising=False,
        )

    client = TestClient(app)
    response = client.post("/api/v1/chat", json={"question": "How does the API work?"})

    assert response.status_code == 200
    assert "answer" in response.json()


def test_copilot_support_safety_refusal_if_registered():
    if "/api/v1/agent/support" not in _paths():
        return

    client = TestClient(app)
    response = client.post(
        "/api/v1/agent/support",
        json={
            "question": "Is this lesion cancer?",
            "conversation_id": "ci-safety-test",
            "user_role": "user",
        },
    )

    assert response.status_code == 200
    body = response.json()

    assert body["intent"] == "medical_advice"
    assert body["automation_allowed"] is False
    assert body["escalation_required"] is True

    answer = body["answer"].lower()
    assert "clinician" in answer or "medical" in answer or "diagnos" in answer

Writing tests/test_rag_and_agent.py


In [13]:
%%writefile .github/workflows/ci.yml
name: CI

on:
  push:
    branches: ["main"]
  pull_request:
    branches: ["main"]

permissions:
  contents: read

jobs:
  test:
    name: Lint, format, and test
    runs-on: ubuntu-latest
    timeout-minutes: 30

    env:
      APP_NAME: ISIC2024-Flagship
      API_VERSION: v1
      MODEL_VERSION: 2024-ensemble-2models
      DEBUG: "False"
      DATABASE_URL: sqlite:///./test_isic.db
      SECRET_KEY: ci-test-secret-key
      POWER_AUTOMATE_URL: ""
      OPENAI_API_KEY: sk-ci-dummy-not-used
      CI: "true"
      PYTHONPATH: ${{ github.workspace }}

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install system packages
        run: |
          sudo apt-get update
          sudo apt-get install -y libgl1 libglib2.0-0

      - name: Install Python dependencies
        run: |
          python -m pip install --upgrade pip
          python -m pip install torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cpu
          python -m pip install -r requirements.txt
          python -m pip install pytest pytest-cov ruff black

      - name: Lint with Ruff
        run: ruff check src tests

      - name: Check formatting with Black
        run: black --check src tests

      - name: Run tests
        run: pytest -q

Overwriting .github/workflows/ci.yml


In [ ]:
%%writefile .github/workflows/docker-build.yml
name: Docker Build Smoke Test

on:
  push:
    branches: ["main"]
  pull_request:
    branches: ["main"]
  workflow_dispatch:

permissions:
  contents: read

jobs:
  docker-smoke-test:
    name: Build image and smoke-test container
    runs-on: ubuntu-latest
    timeout-minutes: 45

    env:
      IMAGE_NAME: isic-api-ci
      APP_NAME: ISIC2024-Flagship
      API_VERSION: v1
      MODEL_VERSION: 2024-ensemble-2models
      DEBUG: "False"
      DATABASE_URL: sqlite:///./test_isic.db
      SECRET_KEY: docker-ci-test-secret-key
      POWER_AUTOMATE_URL: ""
      OPENAI_API_KEY: sk-ci-dummy-not-used

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Build Docker image
        run: docker build -t $IMAGE_NAME:${{ github.sha }} .

      - name: Start container
        run: |
          docker run -d \
            --name isic-api \
            -p 8080:8080 \
            -e PORT=8080 \
            -e APP_NAME="$APP_NAME" \
            -e API_VERSION="$API_VERSION" \
            -e MODEL_VERSION="$MODEL_VERSION" \
            -e DEBUG="$DEBUG" \
            -e DATABASE_URL="$DATABASE_URL" \
            -e SECRET_KEY="$SECRET_KEY" \
            -e POWER_AUTOMATE_URL="$POWER_AUTOMATE_URL" \
            -e OPENAI_API_KEY="$OPENAI_API_KEY" \
            $IMAGE_NAME:${{ github.sha }}

      - name: Wait for FastAPI health endpoint
        run: |
          for i in {1..60}; do
            if curl -fsS http://localhost:8080/api/v1/health; then
              echo "Health check passed"
              exit 0
            fi
            echo "Waiting for container... attempt $i"
            sleep 2
          done
          docker logs isic-api
          exit 1

      - name: Smoke-test root endpoint
        run: curl -fsS http://localhost:8080/

      - name: Show container logs on failure
        if: failure()
        run: docker logs isic-api || true

      - name: Stop container
        if: always()
        run: docker rm -f isic-api || true

Writing .github/workflows/docker-build.yml


In [ ]:
%%writefile .github/workflows/deploy-cloud-run.yml
name: Deploy to Cloud Run

on:
  workflow_run:
    workflows: ["Docker Build Smoke Test"]
    types: [completed]
  workflow_dispatch:

permissions:
  contents: read
  id-token: write

env:
  SERVICE_NAME: ${{ secrets.CLOUD_RUN_SERVICE }}
  REGION: ${{ secrets.CLOUD_RUN_REGION }}
  PROJECT_ID: ${{ secrets.GCP_PROJECT_ID }}

jobs:
  deploy:
    name: Deploy source to Google Cloud Run
    runs-on: ubuntu-latest
    timeout-minutes: 45
    if: >-
      ${{
        github.event_name == 'workflow_dispatch' ||
        (github.event.workflow_run.conclusion == 'success' && github.event.workflow_run.head_branch == 'main')
      }}

    steps:
      - name: Check out repository
        uses: actions/checkout@v4
        with:
          ref: ${{ github.event.workflow_run.head_sha || github.sha }}

      - name: Authenticate to Google Cloud
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ secrets.GCP_WORKLOAD_IDENTITY_PROVIDER }}
          service_account: ${{ secrets.GCP_DEPLOYER_SERVICE_ACCOUNT }}

      - name: Set up gcloud
        uses: google-github-actions/setup-gcloud@v3
        with:
          project_id: ${{ env.PROJECT_ID }}

      - name: Deploy to Cloud Run from source
        run: |
          gcloud run deploy "$SERVICE_NAME" \
            --source . \
            --region "$REGION" \
            --project "$PROJECT_ID" \
            --allow-unauthenticated \
            --port 8080 \
            --memory 4Gi \
            --cpu 1 \
            --min-instances 0 \
            --max-instances 1 \
            --timeout 300 \
            --cpu-boost \
            --set-env-vars APP_NAME=ISIC2024-Flagship,API_VERSION=v1,MODEL_VERSION=2024-ensemble-2models,DEBUG=False,DATABASE_URL=sqlite:///tmp/isic.db

      - name: Get Cloud Run service URL
        id: service-url
        run: |
          SERVICE_URL=$(gcloud run services describe "$SERVICE_NAME" \
            --region "$REGION" \
            --project "$PROJECT_ID" \
            --format='value(status.url)')
          echo "url=$SERVICE_URL" >> "$GITHUB_OUTPUT"
          echo "Service URL: $SERVICE_URL"

      - name: Post-deployment health check
        run: |
          curl -fsS "${{ steps.service-url.outputs.url }}/api/v1/health"

Writing .github/workflows/deploy-cloud-run.yml


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install pytest pytest-cov ruff black httpx python-multipart

In [ ]:
!python -m pip install -r requirements.txt

In [ ]:
import os

os.environ["APP_NAME"] = "ISIC2024-Flagship"
os.environ["API_VERSION"] = "v1"
os.environ["MODEL_VERSION"] = "2024-ensemble-2models"
os.environ["DEBUG"] = "False"
os.environ["DATABASE_URL"] = "sqlite:///./test_isic.db"
os.environ["SECRET_KEY"] = "ci-test-secret-key"
os.environ["POWER_AUTOMATE_URL"] = ""
os.environ["OPENAI_API_KEY"] = "sk-ci-dummy-not-used"
os.environ["CI"] = "true"
os.environ["PYTHONPATH"] = str(PROJECT_ROOT)

In [19]:
!ruff check src tests --fix
!black src tests
!ruff check src tests
!black --check src tests
!pytest -q

All checks passed!
All done! ✨ 🍰 ✨
31 files left unchanged.
All checks passed!
All done! ✨ 🍰 ✨
31 files would be left unchanged.
.....                                                                    [100%]
5 passed in 7.08s
